# MODULE 1.1: Lakebase Instance Creation

This notebook creates a Databricks Lakebase PostgreSQL instance for the Personal Expense Tracker application.

## Objectives
- Create a Lakebase PostgreSQL instance using Databricks SDK
- Configure proper naming convention
- Set up region configuration
- Optimize storage size for development
- Select appropriate database version
- Retrieve and store connection details
- Enable SSL connection
- Implement error handling


In [ ]:
# Install required packages
%pip install databricks-sdk --quiet
%pip install python-dotenv --quiet


In [ ]:
import json
import os
import time
from datetime import datetime
from pathlib import Path
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import CreateLakebaseInstanceRequest, InstanceSize, InstanceStatus
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize Databricks workspace client
w = WorkspaceClient()

print("✓ Databricks SDK initialized successfully")


## Configuration

Define instance parameters following best practices for development environment.


In [ ]:
# Configuration parameters
INSTANCE_NAME = "personal-expense-tracker-db"
REGION = "us-east-1"  # Change to your preferred region
INSTANCE_SIZE = InstanceSize.SMALL  # Small instance for development
STORAGE_SIZE_GB = 20  # Optimized for development (minimum recommended)
DATABASE_VERSION = "15"  # PostgreSQL 15 (latest stable)
ENABLE_SSL = True

# Project paths
PROJECT_ROOT = Path("/Workspace/Repos/Personal Expense Tracker")
CONFIG_DIR = PROJECT_ROOT / "configuration"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Instance Name: {INSTANCE_NAME}")
print(f"Region: {REGION}")
print(f"Instance Size: {INSTANCE_SIZE}")
print(f"Storage Size: {STORAGE_SIZE_GB} GB")
print(f"Database Version: PostgreSQL {DATABASE_VERSION}")
print(f"SSL Enabled: {ENABLE_SSL}")


## Check Existing Instances

Verify if an instance with the same name already exists to avoid duplicates.


In [ ]:
def check_existing_instance(instance_name: str) -> bool:
    """Check if a Lakebase instance with the given name already exists."""
    try:
        instances = w.lakebase.list_instances()
        for instance in instances:
            if instance.name == instance_name:
                print(f"⚠️  Instance '{instance_name}' already exists with ID: {instance.instance_id}")
                return True
        print(f"✓ No existing instance found with name '{instance_name}'")
        return False
    except Exception as e:
        print(f"⚠️  Error checking existing instances: {str(e)}")
        return False

# Check for existing instance
instance_exists = check_existing_instance(INSTANCE_NAME)


## Create Lakebase Instance

Create a new PostgreSQL instance with optimized settings for development.


In [ ]:
def create_lakebase_instance(
    name: str,
    region: str,
    instance_size: InstanceSize,
    storage_size_gb: int,
    database_version: str,
    enable_ssl: bool = True
) -> dict:
    """
    Create a Lakebase PostgreSQL instance.
    
    Args:
        name: Instance name (following naming convention)
        region: AWS region for deployment
        instance_size: Instance size (SMALL, MEDIUM, LARGE)
        storage_size_gb: Storage size in GB
        database_version: PostgreSQL version
        enable_ssl: Enable SSL connections
    
    Returns:
        dict: Instance creation response with connection details
    """
    try:
        print(f"\n🚀 Creating Lakebase instance: {name}")
        print(f"   Region: {region}")
        print(f"   Size: {instance_size}")
        print(f"   Storage: {storage_size_gb} GB")
        print(f"   PostgreSQL Version: {database_version}")
        
        # Create instance request
        request = CreateLakebaseInstanceRequest(
            name=name,
            region=region,
            instance_size=instance_size,
            storage_size_gb=storage_size_gb,
            database_version=database_version,
            enable_ssl=enable_ssl
        )
        
        # Create the instance
        instance = w.lakebase.create_instance(request)
        
        print(f"\n✓ Instance creation initiated successfully!")
        print(f"   Instance ID: {instance.instance_id}")
        print(f"   Status: {instance.status}")
        
        return {
            "instance_id": instance.instance_id,
            "name": instance.name,
            "status": instance.status,
            "region": instance.region,
            "created_at": datetime.now().isoformat()
        }
        
    except Exception as e:
        error_msg = f"❌ Error creating instance: {str(e)}"
        print(error_msg)
        raise Exception(error_msg)


In [ ]:
# Create instance if it doesn't exist
if not instance_exists:
    instance_info = create_lakebase_instance(
        name=INSTANCE_NAME,
        region=REGION,
        instance_size=INSTANCE_SIZE,
        storage_size_gb=STORAGE_SIZE_GB,
        database_version=DATABASE_VERSION,
        enable_ssl=ENABLE_SSL
    )
else:
    print("\n⏭️  Skipping instance creation (already exists)")
    # Get existing instance info
    instances = w.lakebase.list_instances()
    for instance in instances:
        if instance.name == INSTANCE_NAME:
            instance_info = {
                "instance_id": instance.instance_id,
                "name": instance.name,
                "status": instance.status,
                "region": instance.region,
                "created_at": "existing"
            }
            break


In [ ]:
def wait_for_instance_ready(instance_id: str, max_wait_minutes: int = 15) -> bool:
    """
    Wait for the instance to be in READY status.
    
    Args:
        instance_id: The instance ID to check
        max_wait_minutes: Maximum time to wait in minutes
    
    Returns:
        bool: True if instance is ready, False if timeout
    """
    print(f"\n⏳ Waiting for instance to be ready...")
    
    start_time = time.time()
    max_wait_seconds = max_wait_minutes * 60
    
    while True:
        try:
            instance = w.lakebase.get_instance(instance_id)
            status = instance.status
            
            elapsed = int(time.time() - start_time)
            print(f"   Status: {status} (elapsed: {elapsed}s)", end="\r")
            
            if status == InstanceStatus.READY:
                print(f"\n✓ Instance is READY! (took {elapsed}s)")
                return True
            elif status == InstanceStatus.FAILED:
                print(f"\n❌ Instance creation FAILED")
                return False
            
            if time.time() - start_time > max_wait_seconds:
                print(f"\n⚠️  Timeout waiting for instance (>{max_wait_minutes} minutes)")
                return False
            
            time.sleep(10)  # Check every 10 seconds
            
        except Exception as e:
            print(f"\n⚠️  Error checking instance status: {str(e)}")
            time.sleep(10)

# Wait for instance to be ready
if 'instance_info' in locals():
    instance_ready = wait_for_instance_ready(instance_info["instance_id"])
else:
    print("⚠️  No instance info available to check")
    instance_ready = False


## Retrieve Connection Details

Get connection information including host, port, database name, and credentials.


In [ ]:
def get_connection_details(instance_id: str) -> dict:
    """
    Retrieve connection details for the Lakebase instance.
    
    Args:
        instance_id: The instance ID
    
    Returns:
        dict: Connection details including host, port, database, credentials
    """
    try:
        instance = w.lakebase.get_instance(instance_id)
        
        # Get connection endpoint
        endpoint = instance.endpoint
        
        # Get credentials (if available)
        credentials = None
        try:
            creds = w.lakebase.get_instance_credentials(instance_id)
            credentials = {
                "username": creds.username,
                "password": creds.password  # Store securely in production
            }
        except Exception as e:
            print(f"⚠️  Could not retrieve credentials: {str(e)}")
        
        connection_details = {
            "instance_id": instance_id,
            "host": endpoint.host if endpoint else None,
            "port": endpoint.port if endpoint else 5432,
            "database": "postgres",  # Default database
            "ssl_mode": "require" if ENABLE_SSL else "disable",
            "credentials": credentials,
            "connection_string": None
        }
        
        # Build connection string
        if connection_details["host"] and credentials:
            conn_str = (
                f"postgresql://{credentials['username']}:{credentials['password']}@"
                f"{connection_details['host']}:{connection_details['port']}/"
                f"{connection_details['database']}?sslmode={connection_details['ssl_mode']}"
            )
            connection_details["connection_string"] = conn_str
        
        return connection_details
        
    except Exception as e:
        error_msg = f"❌ Error retrieving connection details: {str(e)}"
        print(error_msg)
        raise Exception(error_msg)

# Get connection details
if instance_ready and 'instance_info' in locals():
    connection_details = get_connection_details(instance_info["instance_id"])
    
    print("\n" + "="*60)
    print("CONNECTION DETAILS")
    print("="*60)
    print(f"Host: {connection_details['host']}")
    print(f"Port: {connection_details['port']}")
    print(f"Database: {connection_details['database']}")
    print(f"SSL Mode: {connection_details['ssl_mode']}")
    if connection_details['credentials']:
        print(f"Username: {connection_details['credentials']['username']}")
        print(f"Password: {'*' * len(connection_details['credentials']['password'])}")
    print("="*60)
else:
    print("\n⚠️  Instance not ready. Cannot retrieve connection details.")
    connection_details = None


In [ ]:
def save_lakebase_config(instance_info: dict, connection_details: dict, config_path: Path):
    """
    Save Lakebase configuration to a JSON file.
    
    Args:
        instance_info: Instance metadata
        connection_details: Connection details
        config_path: Path to save configuration file
    """
    try:
        config = {
            "instance": {
                "instance_id": instance_info["instance_id"],
                "name": instance_info["name"],
                "region": instance_info["region"],
                "status": instance_info["status"],
                "created_at": instance_info["created_at"]
            },
            "connection": connection_details if connection_details else {},
            "configuration": {
                "instance_size": str(INSTANCE_SIZE),
                "storage_size_gb": STORAGE_SIZE_GB,
                "database_version": DATABASE_VERSION,
                "ssl_enabled": ENABLE_SSL
            }
        }
        
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)
        
        print(f"\n✓ Configuration saved to: {config_path}")
        
    except Exception as e:
        print(f"⚠️  Error saving configuration: {str(e)}")

# Save configuration
if 'instance_info' in locals():
    config_path = CONFIG_DIR / "lakebase-config.json"
    save_lakebase_config(instance_info, connection_details if 'connection_details' in locals() else None, config_path)
    
    # Display saved config
    print("\n📄 Saved Configuration:")
    with open(config_path, 'r') as f:
        saved_config = json.load(f)
        print(json.dumps(saved_config, indent=2))
else:
    print("\n⚠️  No instance info to save")


## Test Connection (Optional)

Test the database connection to verify everything is working correctly.


In [ ]:
# Optional: Test connection
# Uncomment the following code to test the connection

# %pip install psycopg2-binary --quiet
# import psycopg2
# 
# if connection_details and connection_details.get('connection_string'):
#     try:
#         conn = psycopg2.connect(connection_details['connection_string'])
#         cursor = conn.cursor()
#         cursor.execute("SELECT version();")
#         version = cursor.fetchone()[0]
#         print(f"\n✓ Connection test successful!")
#         print(f"   PostgreSQL Version: {version}")
#         cursor.close()
#         conn.close()
#     except Exception as e:
#         print(f"\n❌ Connection test failed: {str(e)}")
# else:
#     print("\n⚠️  Connection details not available for testing")


## Summary

✅ **Lakebase Instance Created Successfully**

The PostgreSQL instance is now ready for the Personal Expense Tracker application.

### Next Steps:
1. Proceed to `02-create-schema.ipynb` to create database tables
2. Configure your application to use the connection details from `configuration/lakebase-config.json`
3. Keep credentials secure - consider using Databricks Secrets for production

### Important Notes:
- Instance is configured for development (SMALL size, 20GB storage)
- SSL is enabled for secure connections
- Connection details are saved in `configuration/lakebase-config.json`
- For production, scale up instance size and storage as needed
